# Notebook 4: Quantum annealer với MAX-SAT (Hình 4, Bảng 2, Ví dụ 15–16)

Đi kèm [Phần A, Mục 5](../../docs/A-translation/05-quantum-annealer.md). Notebook này **không cần tài khoản D-Wave**: ta dùng `ExactSolver` (liệt kê mọi trạng thái) và `SimulatedAnnealingSampler` của `dimod` thay cho máy thật.

Mục tiêu chính là làm rõ lỗi trong Hình 4 của bài gốc: hàm phạt trong code thiếu số hạng $-2x_3 z$ so với phương trình (39).

In [1]:
import itertools, sys
from collections import Counter
from dimod import Binary, ExactSolver, SimulatedAnnealingSampler
sys.path.insert(0, "../../code")
from qco_utils import penalty_constant

x1, x2, x3, z = (Binary(v) for v in ("x1", "x2", "x3", "z"))
P = 7
# Phần mục tiêu sau khi giảm bậc bằng z = x2*x3 (Ví dụ 15)
base = x1*z - 2*z - x1*x3 + x2 + x3
penalty_paper = x2*x3 - 2*x2*z + 3*z                 # Hình 4 trong bài gốc (thiếu -2*x3*z)
penalty_fixed = x2*x3 - 2*x2*z - 2*x3*z + 3*z        # phương trình (39)
model_paper = base + P * penalty_paper
model_fixed = base + P * penalty_fixed

## 1. Hằng số phạt

Phần mục tiêu sau khi giảm bậc có các hệ số $1, -2, -1, 1, 1$ (cho $x_1 z$, $z$, $x_1 x_3$, $x_2$, $x_3$), nên theo (36) $P = 1 + 6 = 7$, đúng với giá trị bài báo dùng.

In [2]:
reduced = {frozenset({0, 3}): 1, frozenset({3}): -2, frozenset({0, 2}): -1, frozenset({1}): 1, frozenset({2}): 1}
print("P theo (36) =", penalty_constant(reduced))

P theo (36) = 7


## 2. Các trạng thái năng lượng thấp nhất (ExactSolver)

`ExactSolver` liệt kê cả 16 tổ hợp của $(x_1, x_2, x_3, z)$.

In [3]:
def ground_states(model):
    res = ExactSolver().sample(model)
    lowest = res.lowest()
    out = sorted(tuple(int(s[v]) for v in ("x1", "x2", "x3", "z")) for s in lowest.samples())
    return float(lowest.first.energy), out

# Các phép gán thỏa mọi clause của MAX-SAT (Ví dụ 2), tính trực tiếp từ các clause
def count_unsat(a, b, c):
    clauses = [(a or (not b) or c), (a or b or (not c)), ((not a) or (not b) or c)]
    return sum(1 for cl in clauses if not cl)
sat = [t for t in itertools.product((0, 1), repeat=3) if count_unsat(*t) == 0]
print("Phép gán thỏa mọi clause (3 biến):", sat, f"-> {len(sat)} phép gán")

for name, model in (("Phương trình (39), đã sửa", model_fixed), ("Hình 4 gốc, thiếu -2*x3*z", model_paper)):
    e, gs = ground_states(model)
    print(f"\n{name}: năng lượng cực tiểu = {e}, số trạng thái cực tiểu = {len(gs)}")
    for g in gs:
        print("   (x1,x2,x3,z) =", g, "| (x1,x2,x3) =", g[:3], "| z == x2*x3 ?", g[3] == g[1]*g[2])

Phép gán thỏa mọi clause (3 biến): [(0, 0, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 1)] -> 5 phép gán

Phương trình (39), đã sửa: năng lượng cực tiểu = 0.0, số trạng thái cực tiểu = 5
   (x1,x2,x3,z) = (0, 0, 0, 0) | (x1,x2,x3) = (0, 0, 0) | z == x2*x3 ? True
   (x1,x2,x3,z) = (0, 1, 1, 1) | (x1,x2,x3) = (0, 1, 1) | z == x2*x3 ? True
   (x1,x2,x3,z) = (1, 0, 0, 0) | (x1,x2,x3) = (1, 0, 0) | z == x2*x3 ? True
   (x1,x2,x3,z) = (1, 0, 1, 0) | (x1,x2,x3) = (1, 0, 1) | z == x2*x3 ? True
   (x1,x2,x3,z) = (1, 1, 1, 1) | (x1,x2,x3) = (1, 1, 1) | z == x2*x3 ? True

Hình 4 gốc, thiếu -2*x3*z: năng lượng cực tiểu = 0.0, số trạng thái cực tiểu = 3
   (x1,x2,x3,z) = (0, 0, 0, 0) | (x1,x2,x3) = (0, 0, 0) | z == x2*x3 ? True
   (x1,x2,x3,z) = (1, 0, 0, 0) | (x1,x2,x3) = (1, 0, 0) | z == x2*x3 ? True
   (x1,x2,x3,z) = (1, 0, 1, 0) | (x1,x2,x3) = (1, 0, 1) | z == x2*x3 ? True


- **Phương trình (39) / mã đã sửa:** đúng 5 trạng thái cực tiểu, ứng với đúng 5 phép gán thỏa mọi clause; hai trong số đó (`011` và `111`) có $z = 1$.
- **Mã trong Hình 4 gốc:** chỉ còn 3 trạng thái cực tiểu (`000`, `100`, `101`). Phép gán `011` và `111` bị loại vì với $x_2 = x_3 = z = 1$ hàm phạt bị tính sai thành $2P$ thay vì 0.

Vì vậy Bảng 2 của bài báo (chỉ có ba phép gán Energy 0) phản ánh QUBO bị thiếu số hạng, không phải toàn bộ lời giải của MAX-SAT.

## 3. Lấy mẫu bằng simulated annealing (thay cho máy thật)

Đây là tương tự Bảng 2: mỗi hàng là một tổ hợp quan sát được qua 1000 lần chạy. Cột `unsat` là số clause không được thỏa tại $(x_1, x_2, x_3)$.

In [4]:
def sample_table(model, reads=1000):
    res = SimulatedAnnealingSampler().sample(model, num_reads=reads).aggregate()
    rows = []
    for rec in res.data(["sample", "energy", "num_occurrences"]):
        s = rec.sample
        rows.append((int(s["x1"]), int(s["x2"]), int(s["x3"]), int(s["z"]), float(rec.energy),
                     count_unsat(int(s["x1"]), int(s["x2"]), int(s["x3"])), int(rec.num_occurrences)))
    return sorted(rows, key=lambda r: (r[4], -r[6]))

for name, model in (("Mã đã sửa (39)", model_fixed), ("Hình 4 gốc", model_paper)):
    print(name)
    print("  x1 x2 x3 z  energy  unsat  occurrences")
    for r in sample_table(model)[:8]:
        print(f"   {r[0]}  {r[1]}  {r[2]}  {r[3]}  {r[4]:6.1f}  {r[5]:5d}  {r[6]:11d}")
    print()

Mã đã sửa (39)
  x1 x2 x3 z  energy  unsat  occurrences


   0  0  0  0     0.0      0          242
   1  0  1  0     0.0      0          238
   1  0  0  0     0.0      0          234
   1  1  1  1     0.0      0          158
   0  1  1  1     0.0      0          128

Hình 4 gốc
  x1 x2 x3 z  energy  unsat  occurrences


   0  0  0  0     0.0      0          346
   1  0  1  0     0.0      0          344
   1  0  0  0     0.0      0          310



Với mã đã sửa, các trạng thái Energy $= 0$ gồm cả `011` và `111` (với $z = 1$). Với mã gốc chúng không bao giờ xuất hiện ở mức năng lượng thấp nhất, khớp với Bảng 2 của bài báo. (Số lần xuất hiện cụ thể phụ thuộc vào sampler và hạt giống ngẫu nhiên, nên không trùng với số liệu trên phần cứng D-Wave.)

## 4. Biến đổi cục bộ (Ví dụ 16)

Ta cực tiểu hóa $-f(x)$ với $f$ ở Ví dụ 10. Số hạng bậc ba $-36x_1x_2x_3$ được thay bằng $-36(x_1 + x_2 + x_3 - 2)t$. Kiểm tra: lấy cực tiểu theo $t$ phải cho lại đúng $-f(x)$ tại mọi $x$.

In [5]:
def f(x1, x2, x3):
    zz = -3 + x1 + 2*x2 + 3*x3
    return zz**3 - 6*zz

def f_local(x1, x2, x3, t):                   # phương trình (40), đã sửa x2 + x2 -> x1 + x2
    return 9 - 13*x1 - 14*x2 - 9*x3 + 18*x1*x2 + 18*x1*x3 + 18*x2*x3 - 36*(x1 + x2 + x3 - 2)*t

ok = all(min(f_local(*x, t) for t in (0, 1)) == -f(*x) for x in itertools.product((0, 1), repeat=3))
print("min_t f'(x,t) == -f(x) với mọi x:", ok)

# và nếu dùng nhầm bản in sai (x2 + x2 + x3 - 2) thì không còn đúng:
def f_typo(x1, x2, x3, t):
    return 9 - 13*x1 - 14*x2 - 9*x3 + 18*x1*x2 + 18*x1*x3 + 18*x2*x3 - 36*(x2 + x2 + x3 - 2)*t
print("với bản in sai của bài gốc:", all(min(f_typo(*x, t) for t in (0, 1)) == -f(*x) for x in itertools.product((0, 1), repeat=3)))

min_t f'(x,t) == -f(x) với mọi x: True
với bản in sai của bài gốc: False


## 5. Chạy trên máy D-Wave thật (tùy chọn)

Hướng dẫn tạo tài khoản D-Wave Leap, lấy API token và chạy `EmbeddingComposite(DWaveSampler())` trên phần cứng thật sẽ có ở Phần C (`docs/C-hardware/dwave-leap.md`, đang soạn). Khi chạy bằng mã đã sửa, ta kỳ vọng thấy cả 5 phép gán thỏa mọi clause trong kết quả.